In [1]:
from transformers import GPT2Model, GPT2Tokenizer
import torch

/Users/fcalado/Desktop/anti-trans-legislation/training/heritage/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load pre-trained GPT-2 model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("models/gpt2-heritage-e1")
model = GPT2Model.from_pretrained("models/gpt2-heritage-e1")

# Set model to evaluation mode
model.eval()

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [4]:
# Get word vectors for specific words
def get_word_vector(word, model, tokenizer):
    # Tokenize the word
    tokens = tokenizer.encode(word, return_tensors='pt')
    
    # Get model output (no gradients needed)
    with torch.no_grad():
        outputs = model(tokens)
        # Get the last hidden state (contextualized embeddings)
        hidden_states = outputs.last_hidden_state
        
    # Return the vector for the word (squeeze to remove batch dimension)
    return hidden_states.squeeze(0)

# Example: Get vector for "school"
word_vector = get_word_vector("school", model, tokenizer)
print(f"Vector shape: {word_vector.shape}")
print(f"First 10 dimensions of 'school': {word_vector[0][:10]}")

Vector shape: torch.Size([1, 768])
First 10 dimensions of 'school': tensor([-0.0355, -0.0193, -0.0792, -0.0432, -0.0051, -0.2804,  0.0205,  0.0125,
         0.4872,  0.1279])


In [5]:
# Get embedding matrix (static embeddings before contextualization)
embedding_matrix = model.wte.weight  # Word token embeddings
vocab_size, embedding_dim = embedding_matrix.shape
print(f"Vocabulary size: {vocab_size}")
print(f"Embedding dimension: {embedding_dim}")

# Get embedding for a specific token ID
token_id = tokenizer.encode("school")[0]
static_embedding = embedding_matrix[token_id]
print(f"Static embedding for 'school': {static_embedding[:10]}")

Vocabulary size: 50257
Embedding dimension: 768
Static embedding for 'school': tensor([-0.0603, -0.1634,  0.1067, -0.0378, -0.0331,  0.0958, -0.3101, -0.1817,
        -0.0604,  0.0623], grad_fn=<SliceBackward0>)


In [3]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Function to find similar words based on static embeddings
def find_similar_words(target_word, model, tokenizer, top_k=10):
    # Get the embedding matrix
    embedding_matrix = model.wte.weight.detach().numpy()
    
    # Get token ID for target word
    try:
        target_token_id = tokenizer.encode(target_word, add_special_tokens=False)[0]
    except:
        print(f"Word '{target_word}' not found in vocabulary")
        return []
    
    # Get target word embedding
    target_embedding = embedding_matrix[target_token_id].reshape(1, -1)
    
    # Calculate cosine similarity with all other words
    similarities = cosine_similarity(target_embedding, embedding_matrix)[0]
    
    # Get top-k most similar word indices (excluding the word itself)
    similar_indices = np.argsort(similarities)[::-1][1:top_k+1]
    
    # Convert indices back to words and get their similarities
    similar_words = []
    for idx in similar_indices:
        word = tokenizer.decode([idx])
        similarity = similarities[idx]
        similar_words.append((word, similarity))
    
    return similar_words

# Find words similar to "school"
print("Words most similar to 'school':")
print("-" * 40)
similar_words = find_similar_words("school", model, tokenizer, top_k=15)

for i, (word, similarity) in enumerate(similar_words, 1):
    print(f"{i:2d}. {word:<15} (similarity: {similarity:.4f})")

Words most similar to 'school':
----------------------------------------
 1. School          (similarity: 0.7554)
 2.  school         (similarity: 0.7371)
 3.  School         (similarity: 0.6882)
 4.  SCHOOL         (similarity: 0.6622)
 5.  schools        (similarity: 0.6541)
 6.  Schools        (similarity: 0.6210)
 7.  schooling      (similarity: 0.5698)
 8. chool           (similarity: 0.5673)
 9. fashioned       (similarity: 0.5580)
10. Students        (similarity: 0.5394)
11. College         (similarity: 0.5383)
12. Education       (similarity: 0.5316)
13. student         (similarity: 0.5292)
14.  teachers       (similarity: 0.5236)
15.  university     (similarity: 0.5167)
 1. School          (similarity: 0.7554)
 2.  school         (similarity: 0.7371)
 3.  School         (similarity: 0.6882)
 4.  SCHOOL         (similarity: 0.6622)
 5.  schools        (similarity: 0.6541)
 6.  Schools        (similarity: 0.6210)
 7.  schooling      (similarity: 0.5698)
 8. chool           (simi

In [4]:
# Function to find similar words using contextual embeddings
def find_similar_words_contextual(target_word, model, tokenizer, context="", top_k=10):
    """
    Find similar words using contextual embeddings.
    If context is provided, it will be used to generate contextualized embeddings.
    """
    # Prepare the input text
    if context:
        input_text = f"{context} {target_word}"
    else:
        input_text = target_word
    
    # Tokenize the input
    inputs = tokenizer.encode(input_text, return_tensors='pt')
    target_token_pos = -1  # Position of target word (last token if no context)
    
    # Get contextualized embeddings
    with torch.no_grad():
        outputs = model(inputs)
        hidden_states = outputs.last_hidden_state.squeeze(0)  # Remove batch dimension
        target_embedding = hidden_states[target_token_pos].unsqueeze(0)  # Get target word embedding
    
    # Get static embedding matrix for comparison
    embedding_matrix = model.wte.weight.detach().numpy()
    target_embedding_np = target_embedding.detach().numpy()
    
    # Calculate cosine similarity with all vocabulary embeddings
    similarities = cosine_similarity(target_embedding_np, embedding_matrix)[0]
    
    # Get top-k most similar word indices
    similar_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Convert indices back to words and get their similarities
    similar_words = []
    for idx in similar_indices:
        word = tokenizer.decode([idx])
        similarity = similarities[idx]
        similar_words.append((word, similarity))
    
    return similar_words

print("\\nContextual similarity for 'school' (without context):")
print("-" * 50)
contextual_similar = find_similar_words_contextual("school", model, tokenizer, top_k=15)

for i, (word, similarity) in enumerate(contextual_similar, 1):
    print(f"{i:2d}. {word:<15} (similarity: {similarity:.4f})")

print("\\nContextual similarity for 'school' (with educational context):")
print("-" * 60)
contextual_similar_edu = find_similar_words_contextual("school", model, tokenizer, 
                                                       context="The students attended", top_k=15)

for i, (word, similarity) in enumerate(contextual_similar_edu, 1):
    print(f"{i:2d}. {word:<15} (similarity: {similarity:.4f})")

\nContextual similarity for 'school' (without context):
--------------------------------------------------
 1.  of             (similarity: 0.0640)
 2. ,               (similarity: 0.0580)
 3.  is             (similarity: 0.0567)
 4.  in             (similarity: 0.0551)
 5.  and            (similarity: 0.0508)
 6. .               (similarity: 0.0492)
 7.  for            (similarity: 0.0489)
 8.  was            (similarity: 0.0445)
 9.  to             (similarity: 0.0417)
10. :               (similarity: 0.0398)
11.  that           (similarity: 0.0389)
12. -               (similarity: 0.0387)
13.  parents        (similarity: 0.0380)
14.  has            (similarity: 0.0379)
15.  as             (similarity: 0.0375)
\nContextual similarity for 'school' (with educational context):
------------------------------------------------------------
 1.  in             (similarity: -0.0090)
 2. ,               (similarity: -0.0101)
 3.  for            (similarity: -0.0130)
 4.  with           (simil

In [5]:
# Function to explore education-related vocabulary
def explore_education_vocabulary(model, tokenizer, education_words=None, top_k=10):
    """
    Explore words related to education by averaging embeddings of education-related words
    and finding similar words in the vocabulary.
    """
    if education_words is None:
        education_words = ["school", "education", "student", "teacher", "learning", "classroom"]
    
    # Get embeddings for education-related words
    education_embeddings = []
    valid_words = []
    
    for word in education_words:
        try:
            token_id = tokenizer.encode(word, add_special_tokens=False)[0]
            embedding = model.wte.weight[token_id].detach().numpy()
            education_embeddings.append(embedding)
            valid_words.append(word)
        except:
            print(f"Warning: '{word}' not found in vocabulary")
    
    if not education_embeddings:
        return []
    
    # Average the embeddings to get an "education concept" vector
    education_concept = np.mean(education_embeddings, axis=0).reshape(1, -1)
    
    # Get all embeddings
    embedding_matrix = model.wte.weight.detach().numpy()
    
    # Calculate similarities
    similarities = cosine_similarity(education_concept, embedding_matrix)[0]
    
    # Get top similar words
    similar_indices = np.argsort(similarities)[::-1][:top_k * 2]  # Get more to filter out input words
    
    # Filter out the input words and get top_k
    similar_words = []
    for idx in similar_indices:
        word = tokenizer.decode([idx]).strip()
        if word.lower() not in [w.lower() for w in valid_words] and len(similar_words) < top_k:
            similarity = similarities[idx]
            similar_words.append((word, similarity))
    
    return similar_words, valid_words

print("Exploring education-related vocabulary in the fine-tuned model:")
print("=" * 60)
education_related, seed_words = explore_education_vocabulary(model, tokenizer, top_k=20)

print(f"\\nSeed words used: {', '.join(seed_words)}")
print("\\nWords most related to education concepts:")
print("-" * 45)

for i, (word, similarity) in enumerate(education_related, 1):
    print(f"{i:2d}. {word:<20} (similarity: {similarity:.4f})")

Exploring education-related vocabulary in the fine-tuned model:
\nSeed words used: school, education, student, teacher, learning, classroom
\nWords most related to education concepts:
---------------------------------------------
 1. Students             (similarity: 0.6995)
 2. schooling            (similarity: 0.6625)
 3. educating            (similarity: 0.6484)
 4.                     (similarity: 0.6481)
 5. �                    (similarity: 0.6480)
 6. rawdownload          (similarity: 0.6477)
 7. �                    (similarity: 0.6476)
 8.                     (similarity: 0.6475)
 9. 龍�                   (similarity: 0.6473)
10. TheNitrome           (similarity: 0.6473)
11. �                    (similarity: 0.6472)
12.                     (similarity: 0.6472)
13. �                    (similarity: 0.6469)
14. reportprint          (similarity: 0.6469)
15.                      (similarity: 0.6467)
16. quickShip            (similarity: 0.6466)
17.                      (similari